# S4~S7 — 시드 반복 + 패러프레이즈 렌즈 (+ 고친 프로브)

**닫는 것:** `EXP-M01`(single seed, 가장 많이 인용되는 약점) · `LOG-M06`(순응 렌즈 *부류*인가 이 *문자열*인가).

세 가지를 한 세션에 태운다. 셋 다 같은 체크포인트가 필요한데 **Kaggle 체크포인트는 휘발**되므로
(lessons 8) 따로 돌리면 그때마다 재학습해야 한다 — 같이 태우는 게 유일하게 싼 길이다.

1. **시드** 1337 / 2718 → seed 3개. ⚠️표본 3이면 CI가 아니라 **range로 보고**한다.
2. **패러프레이즈 렌즈** `agreeable_para1` — arm1 학습 프롬프트와 의미는 같고 어휘가 다르다.
   여기서도 붕괴하면 방아쇠는 **부류**이고, 안 하면 우리가 잰 것은 문자열에 가깝다. 어느 쪽이든 결과다.
3. **고친 프로브**(`--harm_frame judgement`). S1·S2는 `v_harm`이 렌즈를 안 타서 κ가 정의상 1.0이었고
   GPU 약 20h가 기하 증거를 0개 만들었다. 스크립트에 **회귀 게이트**를 넣었으므로 결함이 재발하면
   즉시 멈춘다.

**세션당 ~2.2h** — 학습 2,400s + 3렌즈 eval 5,400s + 프로브 100s. 4세션 ≈ 8.8h.
**라운드3만 평가한다** — 시드 반복에 필요한 것은 라운드별 곡선이 아니라 최종 효과다.


## 0. Secrets

⚠️ **Secrets는 계정이 아니라 노트북마다 붙는다.** 새 노트북을 만들거나 업로드하면 계정에 등록돼
있어도 안 딸려온다 → **Add-ons → Secrets → `HF_TOKEN` 토글 ON.**

In [ ]:
import os, torch

# 가속기는 시크릿과 성질이 같다 — **노트북마다** 붙고, Import하면 None으로 초기화된다.
# 여기서 안 막으면 패키지 설치를 다 하고 unsloth 내부 스택에서 죽는다.
if not torch.cuda.is_available():
    raise SystemExit(
        "\n[GPU 없음] 이 세션에 가속기가 안 붙었다. unsloth는 CPU에서 로드 자체가 안 된다.\n"
        "  1) 우측 Settings -> Accelerator -> GPU T4 x2 (또는 P100) 선택 후 다시 Run.\n"
        "     노트북을 새로 Import하면 가속기 설정은 따라오지 않는다.\n"
        "  2) 이미 GPU로 돼 있다면 **주간 할당량(30h) 소진**이다 — 우측 상단 게이지를 볼 것.\n"
        "     소진되면 Kaggle은 경고 없이 CPU로 돌린다.\n"
        f"  torch={torch.__version__} cuda_built={torch.version.cuda}"
    )
print(f"GPU: {torch.cuda.device_count()}x {torch.cuda.get_device_name(0)}")

from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise SystemExit(
        "\n[HF_TOKEN 없음] Secrets는 계정이 아니라 **노트북마다** 붙여야 한다.\n"
        "  상단 메뉴 Add-ons -> Secrets -> HF_TOKEN 토글 ON -> 다시 Run\n"
        "  (계정에 이미 등록돼 있어도 새 노트북에는 자동으로 안 딸려온다)\n"
        f"  원본 오류: {e}"
    )
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

## 1. 패키지 설치 (매 세션 필수)
Unsloth 공식 Llama3.1(8B) 노트북 핀: `transformers==4.56.2`, `trl==0.22.2`.

In [ ]:
import subprocess, sys

def pip(*args):
    """`!pip` 매직 대신 subprocess — 이 노트북의 모든 셀이 순수 파이썬이라
    업로드 전에 compile()로 문법 검증을 걸 수 있다(RUNBOOK 함정 15)."""
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("unsloth")
pip("transformers==4.56.2", "trl==0.22.2")
pip("git+https://github.com/dsbowen/strong_reject.git@main")
pip("lm-eval")
print("packages ok")

## 2. 실험 폴더 + 학습 데이터 복구
데이터는 1차 것을 **그대로 재사용**한다 — 다시 만들면 1차와의 비교가 끊긴다.

In [ ]:
import os, shutil, sys, glob

EXP_ROOT  = "/kaggle/working/experiment"
CKPT_ROOT = "/kaggle/tmp/ckpt"   # 휘발성 대용량 (머지 모델 16GB/개)

def _input_tree():
    """붙은 데이터셋을 3단계까지 나열 — 경로가 안 맞을 때 진단용."""
    seen = sorted(glob.glob("/kaggle/input/*")
                  + glob.glob("/kaggle/input/*/*")
                  + glob.glob("/kaggle/input/*/*/*"))
    return seen or ["(비어 있음 — 붙은 데이터셋이 하나도 없다)"]

# 코드 데이터셋 위치 자동 탐색. 마운트 경로가 /kaggle/input/<slug>/ 이기도 하고
# /kaggle/input/datasets/<owner>/<slug>/ 이기도 해서 하드코딩하지 않는다.
if not os.path.exists(EXP_ROOT):
    hits = glob.glob("/kaggle/input/**/src/eval_refusal.py", recursive=True)
    if not hits:
        print("[진단] /kaggle/input 아래 실제 구조:")
        for p in _input_tree():
            print("   ", p)
        raise SystemExit(
            "\n[코드 데이터셋 없음] 'branch-1-premise'가 이 노트북에 안 붙어 있다.\n"
            "  우측 Input 패널 -> Add Input -> branch-1-premise / gen-data-backup 둘 다 추가\n"
            "  (Secrets와 마찬가지로 데이터셋도 노트북마다 따로 붙여야 한다)"
        )
    INPUT_DIR = os.path.dirname(os.path.dirname(hits[0]))
    print("[code] 데이터셋 발견:", INPUT_DIR)
    shutil.copytree(INPUT_DIR, EXP_ROOT)
os.makedirs(CKPT_ROOT, exist_ok=True)
os.chdir(EXP_ROOT)
sys.path.insert(0, os.path.join(EXP_ROOT, "src"))

# zip(압축)과 이미 풀린 폴더(Kaggle 자동 압축해제) 둘 다 지원
if not os.path.exists("data/arm1/round1.jsonl"):
    zip_cands = (glob.glob("/kaggle/input/**/gen_data_backup.zip", recursive=True)
                 + glob.glob("/kaggle/working/**/gen_data_backup.zip", recursive=True))
    dir_cands = glob.glob("/kaggle/input/**/arm1/round1.jsonl", recursive=True)
    if zip_cands:
        shutil.unpack_archive(zip_cands[0], "data")
        print("[data] 복구(zip):", zip_cands[0])
    elif dir_cands:
        src_root = os.path.dirname(os.path.dirname(dir_cands[0]))
        shutil.copytree(src_root, "data", dirs_exist_ok=True)
        print("[data] 복구(폴더):", src_root)
    else:
        print("[진단] /kaggle/input 아래 실제 구조:")
        for p in _input_tree():
            print("   ", p)
        raise SystemExit(
            "\n[학습 데이터 없음] 'gen-data-backup'이 이 노트북에 안 붙어 있다.\n"
            "  우측 Input 패널 -> Add Input -> gen-data-backup 추가"
        )

print("cwd:", os.getcwd())
print("src:", sorted(os.listdir("src")))
print("data arms:", sorted(os.path.basename(d) for d in glob.glob("data/arm*")))

# R2 코드가 맞는지 게이트 — 하나라도 실패하면 1차 코드가 올라가 있는 것이다
for path, needle in [("src/eval_refusal.py", "schema"),
                     ("src/train_round.py", "frac_ckpt"),
                     ("src/probe_directions.py", "harm_frame"),
                     ("configs/arms.json", "eval_lenses")]:
    assert os.path.exists(path), f"R2 코드 아님: {path} 없음 → Dataset New Version 재업로드 필요"
    if needle:
        assert needle in open(path, encoding="utf-8").read(), \
            f"R2 코드 아님: {path}에 '{needle}' 없음 → Dataset New Version 재업로드 필요"
print("[gate] R2 코드 확인됨")

## 3. 세션 함수 정의

In [ ]:
import json, shutil, subprocess, os

BASE = "unsloth/Meta-Llama-3.1-8B-Instruct"
FULL = " --full"       # R2는 전 라운드 n=313. 1차의 n 불일치가 라운드 비교를 막았다.
MMLU_LIMIT = "30"

def run(cmd: str):
    print(f"\n$ {cmd}", flush=True)
    r = subprocess.run(cmd, shell=True)
    assert r.returncode == 0, f"실패: {cmd}"

def check(path, min_kb=20):
    """per-prompt 레코드가 실제로 쓰였는지 즉시 확인 — 1차의 핵심 실패를 막는 게이트."""
    kb = os.path.getsize(path) / 1024
    print(f"[check] {os.path.basename(path)} = {kb:.0f} KB")
    assert kb > min_kb, f"per-prompt 레코드가 비었다: {path}"

def probe_baseline(seed, harm_frame="judgement"):
    """동결기저(θ_0, ∅). 학습 전에 한 번. 이후 프로브가 이걸 기준으로 읽는다.
       ⚠️기저와 이후 라운드의 harm_frame이 다르면 ρ·D가 무의미해진다 — 같은 값을 쓸 것."""
    run(f"python src/probe_directions.py --model_path {BASE} --arm arm0 --round 0 --seed {seed} "
        f"--harm_frame {harm_frame}")
    return f"results/probe_arm0_r0_s{seed}.json"

def run_session(arm, seed, lenses="none,agreeable,principled",
                probe=False, frac=False, mmlu=True, eval_rounds=(1, 2, 3),
                harm_frame="judgement", mmlu_limit=None, free_ckpt=True):
    """★2026-07-29 확장 3건:
       eval_rounds  — 학습은 순차라 건너뛸 수 없지만 **평가는 건너뛸 수 있다.**
                      비용의 거의 전부가 eval(1렌즈 n=313 = ~1,800s)이므로 라운드3만
                      재면 세션이 ~3.4h 짧아진다. 시드 반복은 라운드별 곡선이 아니라
                      최종 효과만 필요하므로 eval_rounds=(3,)이 맞다.
       harm_frame   — 프로브 v_harm 추출 프레이밍. 기본 judgement(고친 것).
                      raw는 S1·S2 재현 전용이며 κ_harm≡1이라 무정보다.
       mmlu_limit   — None이면 전역 MMLU_LIMIT. 논문이 보고하는 대조는 **동일 570문항
                      (limit=10)** 이므로 새 arm은 10으로 도는 편이 비교 가능하고 3배 싸다."""
    base_dirs = probe_baseline(seed, harm_frame) if probe else None
    prev, prev_dir = BASE, None

    for r in [1, 2, 3]:
        ckpt = f"{CKPT_ROOT}/{arm}_r{r}_s{seed}"
        frac_arg = " --frac_ckpt 0.25,0.5" if (frac and r == 1) else ""
        run(f"python src/train_round.py --arm {arm} --round {r} --seed {seed} "
            f"--base_model {prev} --data data/{arm}/round{r}.jsonl "
            f"--output_dir {ckpt}{frac_arg}")

        # 세분 라운드: 라운드1 학습 중 저장된 어댑터를 원본 BASE 위에 얹어 평가
        if frac and r == 1:
            for f in ["0p25", "0p5"]:
                ad = f"{CKPT_ROOT}/{arm}_s{seed}/adapter_frac{f}"
                if not os.path.exists(ad):
                    print(f"[warn] 세분 체크포인트 없음: {ad}")
                    continue
                rn = "0.25" if f == "0p25" else "0.5"
                run(f"python src/eval_refusal.py --model_path {BASE} --adapter_path {ad} "
                    f"--arm {arm} --round {rn} --seed {seed} --lenses {lenses}{FULL}")
                check(f"results/eval_{arm}_r{f}_s{seed}.json")
                if probe:
                    run(f"python src/probe_directions.py --model_path {BASE} --adapter_path {ad} "
                        f"--arm {arm} --round {rn} --seed {seed} --baseline_dirs {base_dirs}")

        if r in eval_rounds:
            run(f"python src/eval_refusal.py --model_path {ckpt} --arm {arm} --round {r} "
                f"--seed {seed} --lenses {lenses}{FULL}")
            check(f"results/eval_{arm}_r{r}_s{seed}.json")
        else:
            print(f"[skip] eval arm={arm} r={r} — eval_rounds={eval_rounds} (비용 절감)")

        if probe and r in eval_rounds:
            run(f"python src/probe_directions.py --model_path {ckpt} --arm {arm} --round {r} "
                f"--seed {seed} --baseline_dirs {base_dirs} --harm_frame {harm_frame}")

        if mmlu and r == 3:
            run(f"bash src/run_mmlu.sh {ckpt} {arm} {r} {mmlu_limit or MMLU_LIMIT}")

        # 디스크: 머지본 1개 = 16GB. 다음 라운드 전에 이전 것 삭제.
        if prev_dir and os.path.exists(prev_dir):
            shutil.rmtree(prev_dir)
            print(f"[disk] 삭제: {prev_dir}")
        prev, prev_dir = ckpt, ckpt

    # 루프는 '이전' 라운드만 지운다 — 마지막 라운드 머지본 16GB가 그대로 남는다.
    # 한 세션만 돌면 무해하지만, 세션을 이어 돌리면 세션당 16GB씩 쌓여
    # 시간이 아니라 디스크가 먼저 터진다. MMLU는 위 루프 안에서 이미 끝났다.
    if free_ckpt and prev_dir and os.path.exists(prev_dir):
        shutil.rmtree(prev_dir)
        print(f"[disk] 삭제(최종): {prev_dir}")
        return None
    return prev

def run_baseline(seed, lenses="none,agreeable,principled", mmlu_limit=None):
    """arm0 = 원본 모델. 학습 없음. 페르소나 방아쇠 자체가 무해함을 보이는 통제군."""
    run(f"python src/eval_refusal.py --model_path {BASE} --arm arm0 --round 0 "
        f"--seed {seed} --lenses {lenses}{FULL}")
    check(f"results/eval_arm0_r0_s{seed}.json")
    run(f"bash src/run_mmlu.sh {BASE} arm0 0 {mmlu_limit or MMLU_LIMIT}")

print("정의 완료")

## 4. ★이번 세션 — 한 줄만 살린다

In [ ]:
LENSES = "none,agreeable,agreeable_para1"

# ★★ 커밋마다 이 값 하나만 바꾼다. 1회차 = S4+S5, 2회차 = S6+S7. ★★
# 줄을 주석 처리해서 고르지 않는다 — 셀 부분 편집으로 줄을 잘못 살리는 것이
# 이 프로젝트에서 이미 나온 실패 방식이다.
BATCH = 1

BATCHES = {
    1: [("S4", "arm1", 1337), ("S5", "arm3", 1337)],   # 시드 1337 — 이걸로 시드가 2개가 된다
    2: [("S6", "arm1", 2718), ("S7", "arm3", 2718)],   # 시드 2718 — 3개째
}
assert BATCH in BATCHES, f"BATCH는 {sorted(BATCHES)} 중 하나여야 한다"
PLAN = BATCHES[BATCH]
print(f"BATCH {BATCH}: " + ", ".join(f"{n}({a} x {s})" for n, a, s in PLAN))


def snap(tag):
    """세션이 끝날 때마다 zip을 떠 둔다. results/는 누적되므로 마지막 zip 하나에
       그 커밋의 전부가 들어 있다. 뒤 세션이 죽어도 앞 세션은 output에 남는다."""
    import shutil, os
    zp = shutil.make_archive(f"/kaggle/working/r2_results_{tag}", "zip",
                             "/kaggle/working/experiment/results")
    print(f"[snap] {zp} | {os.path.getsize(zp)/1e6:.2f} MB", flush=True)


# 커밋당 2세션 x ~2.2h = ~4.4h. 12h 한도에 여유가 크다.
for name, arm, seed in PLAN:
    print(f"\n===== {name}: {arm} x seed {seed} =====", flush=True)
    run_session(arm, seed, lenses=LENSES, eval_rounds=(3,), probe=True, mmlu=False)
    snap(f"{name.lower()}_{arm}_s{seed}")


## 5. ★결과 회수

In [ ]:
import shutil, os, glob

# TAG는 BATCH에서 유도한다 — 손으로 적으면 배치 내용과 어긋난다.
# (1회차 zip이 arm3까지 담고도 `arm1_s1337`로 나왔던 게 그 증상이다.)
TAG = f"batch{BATCH}_s{PLAN[0][2]}"

# snap()이 세션마다 떠 둔 것과 별개인 **그 커밋의 최종본**이다. results/는 누적이므로
# 이 zip 하나에 배치 전부가 들어 있다 — 로컬로 내릴 건 이것 하나면 된다.
zp = shutil.make_archive(f"/kaggle/working/r2_results_{TAG}", "zip",
                         "/kaggle/working/experiment/results")
print("zip:", zp, "| size MB:", round(os.path.getsize(zp)/1e6, 2))
for p in sorted(glob.glob("/kaggle/working/experiment/results/*.json")):
    print(f"  {os.path.getsize(p)/1024:7.0f} KB  {os.path.basename(p)}")


## 6. 세션이 성공했는지 — 특히 프로브가 정보를 담고 있는지

In [ ]:
import json, glob, os, re

# 이 커밋에서 돈 것 전부를 훑는다 — 하나를 하드코딩하면 나머지가 검증 없이 지나간다.
# arm0(동결기저 프로브)는 r0라 잡히지 않는다.
found = sorted(glob.glob("results/probe_arm[1-9]*_r3_s*.json"))
assert found, "probe 결과가 없다 — 세션이 하나도 완주하지 못했다"

for pp in found:
    arm, seed = re.search(r"probe_(arm\d+)_r3_s(\d+)\.json$", os.path.basename(pp)).groups()
    ev = json.load(open(f"results/eval_{arm}_r3_s{seed}.json", encoding="utf-8"))
    for lens, c in ev["conditions"].items():
        assert len(c["records"]) == 313, f"{arm}/s{seed}/{lens}: n={len(c['records'])}"
        print(f"  eval  {arm} s{seed} {lens:>16} refusal={c['refusal_rate']:.3f}")
    pr = json.load(open(pp, encoding="utf-8"))
    kh, kr = pr["readings"]["kappa_v_harm"], pr["readings"]["kappa_v_ref"]
    print(f"  probe {arm} s{seed} kappa_v_ref={kr:.4f} kappa_v_harm={kh:.4f} "
          f"frame={pr.get('harm_frame')}")
    assert pr.get("schema") == "probe_r3", f"{arm}/s{seed}: 옛 프로브 스키마 — Dataset을 새로 올려야 한다"
    assert abs(kh - 1.0) > 1e-4, f"{arm}/s{seed}: K_harm이 1.0 — S1,S2를 무효화한 결함이 재발했다"

# 계획한 만큼 돌았는지. 여기서 걸려도 zip은 snap()이 이미 떠 뒀다 — 회수는 된다.
want = len(PLAN)
assert len(found) == want, (
    f"BATCH {BATCH}는 {want}세션인데 {len(found)}개만 완주했다 — "
    f"zip은 남아 있으니 모자란 세션만 다시 돌려라")

print(f"\n[gate] BATCH {BATCH} 통과 — {len(found)}세션, 프로브가 정보를 담고 있다")
